In [ ]:
import pandas as pd        

# confirm the working directory and file existence
import os
print("cwd:", os.getcwd())
print("exists:", os.path.isfile("personal_finance_dataset.xlsx"))

# read the 'datathon_finance' sheet from the Excel file
df = pd.read_excel("personal_finance_dataset.xlsx", sheet_name="datathon_finance")

column_mapping = {
    'PAGEMIEG': 'Age_Group',
    'PATTCRU':  'Credit_Card_Payment_Behavior',
    'PATTSITC': 'COVID_Financial_Impact',
    'PATTSKP':  'Skipped_Payments',
    'PEDUCMIE': 'Education_Level',
    'PEFATINC': 'Annual_After_Tax_Income',
    'PFMTYPG':  'Family_Type',
    'PFTENUR':  'Home_Ownership',
    'PLFFPTME': 'Work_Status_2022',
    'PNBEARG':  'Number_of_Earners',
    'PPVRES':   'Province',
    'PWAPRVAL': 'Home_Value',
    'PWASTDEP': 'Bank_Deposits',
    'PWATFS':   'TFSA_Balance',
    'PWDPRMOR': 'Mortgage_Debt',
    'PWDSLOAN': 'Student_Loan_Debt',
    'PWDSTCRD': 'Credit_Card_Debt',
    'PWDSTLOC': 'Line_of_Credit_Debt',
    'PWNETWPG': 'Net_Worth'
}

df.rename(columns=column_mapping, inplace=True)

print(df.head())

In [113]:
import numpy as np

def define_resilience(df, shock_percentage=0.15):
    # --------------------------------------------------------------
    # 1. ANNUAL EXPENSE & LIQUIDITY
    df['Total_Liquidity'] = df['Bank_Deposits'] + df['TFSA_Balance']

    # Annual Expense Proxy (60% of income)
    df['Annual_Expenses_Base'] = (df['Annual_After_Tax_Income'] * 0.6)
    
    # Calculate Runway in years (How many years they can cover expenses with liquid assets if income drops to zero)
    df['Years_of_Runway'] = df['Total_Liquidity'] / df['Annual_Expenses_Base'].replace(0, np.nan) 
    
    # --------------------------------------------------------------
    # 2. HOUSING BURDEN
    # Owners (Home_Ownership == 2): Proxy 4% of Mortgage Debt as annual cost
    # Renters (Home_Ownership == 3): Proxy 30% of income as annual rent
    df['Est_Annual_Housing_Cost'] = np.where(
        df['Home_Ownership'] == 2, 
        df['Mortgage_Debt'] * 0.04, 
        np.where(df['Home_Ownership'] == 3, df['Annual_After_Tax_Income'] * 0.30, 0))
    
    # Housing Burden Ratio (Costs / Income)
    df['Housing_Burden_Ratio'] = df['Est_Annual_Housing_Cost'] / df['Annual_After_Tax_Income'].replace(0, np.nan)

    # 3. CONSUMER DEBT BURDEN
    # Calculate Total Non-Mortgage Debt 
    df['Total_Consumer_Debt'] = df['Credit_Card_Debt'] + df['Line_of_Credit_Debt'] + df['Student_Loan_Debt']

    # Annual Debt-to-Income Ratio
    # A ratio > 0.5 is often considered "serious financial strain" in Canadian banking
    df['DTI_Ratio'] = df['Total_Consumer_Debt'] / df['Annual_After_Tax_Income'].replace(0, np.nan)

    # --------------------------------------------------------------
    # 4. THE "AT-RISK" TARGET VARIABLE
    #"At Risk" Baseline (No shock):
    # - Less than 3 months (0.25 yrs) runway
    # - High Housing Burden (>30%)
    # - High DTI Ratio (>50%) 
    # - Skipped Payments
    df['Is_At_Risk_Base'] = np.where(
        (df['Years_of_Runway'] < 0.25) | 
        (df['Housing_Burden_Ratio'] > 0.3) | 
        (df['DTI_Ratio'] > 0.50) |             
        (df['Skipped_Payments'] == 1), 1, 0)
   
    return df

df = define_resilience(df, shock_percentage=0.15)


In [ ]:
# Define a function to apply economic shocks and determine new vulnerability status
def apply_economic_shock(df, shock_type="expenses", severity=0.15):
    temp_df = df.copy()
    
    if shock_type == "expenses":
        # Scenario 1: Inflation Shock (Increase Expenses)
        shocked_expenses = temp_df['Annual_Expenses_Base'] * (1 + severity)
        runway = temp_df['Total_Liquidity'] / shocked_expenses.replace(0, np.nan)
        is_at_risk = (runway < 0.25).astype(int)
        
    elif shock_type == "mortgage":
        # Scenario 2: Housing Shock (Increase Mortgage Costs)
        shocked_mortgage = temp_df['Mortgage_Debt'] * (1 + severity)
        # Re-calc housing burden
        housing_cost = np.where(temp_df['Home_Ownership'] == 2, shocked_mortgage * 0.04, 
                       np.where(temp_df['Home_Ownership'] == 3, temp_df['Annual_After_Tax_Income'] * 0.30, 0))
        burden = housing_cost / temp_df['Annual_After_Tax_Income'].replace(0, np.nan)
        is_at_risk = ((temp_df['Years_of_Runway'] < 0.25) | (burden > 0.3)).astype(int)
        
    elif shock_type == "income":
        # Scenario 3: Recession Shock (Decrease Income)
        shocked_income = temp_df['Annual_After_Tax_Income'] * (1 - severity)
        runway = temp_df['Total_Liquidity'] / shocked_income.replace(0, np.nan)
        is_at_risk = (runway < 0.25).astype(int)
        
    # Return the "Flipped" status: Safe before, At Risk now
    return ((df['Is_At_Risk_Base'] == 0) & (is_at_risk == 1)).astype(int)

df['Vulnerable_Inflation'] = apply_economic_shock(df, "expenses", 0.15)
df['Vulnerable_Mortgage'] = apply_economic_shock(df, "mortgage", 0.15)
df['Vulnerable_Income'] = apply_economic_shock(df, "income", 0.15)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Summary Table by Age Group
age_labels = {1: '18-24', 2: '25-34', 3: '35-44', 4: '45-54'}
summary_table = df.groupby('Age_Group')[['Is_At_Risk_Base', 'Vulnerable_Inflation', 'Vulnerable_Mortgage', 'Vulnerable_Income']].mean() * 100
summary_table.index = summary_table.index.map(age_labels)

print("Financial Fragility by Age Group (%):")
print(summary_table)


In [ ]:
# SHOCK SCENARIO 1: 15% Increase in Expenses (Inflation Shock)
df['Vulnerable_Inflation'] = apply_economic_shock(df, "expenses", 0.15)
shock_summary = df.groupby('Age_Group')[['Is_At_Risk_Base', 'Vulnerable_Inflation']].mean() * 100
plot_data = shock_summary.loc[[1, 2, 3, 4]]

# Chart 1: Bar chart of % newly vulnerable by age group after inflation shock
# Visualize the % of each age group that became vulnerable after the shock
sns.barplot(
    x=['18-24', '25-34', '35-44', '45-54'], 
    y=plot_data['Vulnerable_Inflation'], 
    palette='Reds_r'
)
plt.title('Age Group Breakdown: % Pushed into Risk by 15% Inflation Shock', fontsize=13)
plt.ylabel('% of Group Newly Vulnerable')
plt.show()

# Chart 2: Stacked Bar Chart of Baseline Risk vs. New Risk After Inflation Shock
# Create a stacked bar chart comparing baseline risk vs. new risk after shock for each age group, to answer: "How much did the shock increase vulnerability in each age group?" 
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(plot_data.index, plot_data['Is_At_Risk_Base'], 
       label='Existing Risk (Baseline)', color="#eed66b", alpha=0.8)

ax.bar(plot_data.index, plot_data['Vulnerable_Inflation'], 
       bottom=plot_data['Is_At_Risk_Base'], 
       label='New Risk (After 15% Shock)', color='#e74c3c')

ax.set_xticks([1, 2, 3, 4]) # Set the position of the ticks
ax.set_xticklabels(['18-24', '25-34', '35-44', '45-54']) # Set the text labels

plt.title('Total Financial Vulnerability: Baseline vs. Post Inflation Shock', fontsize=15, fontweight='bold')
plt.ylabel('% of Demographic At-Risk')
plt.xlabel('Age Group')
plt.ylim(0, 80)
plt.legend()

for i, age_idx in enumerate(plot_data.index):
    base_val = plot_data.loc[age_idx, 'Is_At_Risk_Base']
    shock_val = plot_data.loc[age_idx, 'Vulnerable_Inflation']
    total = base_val + shock_val
    
    # Label for the "New Risk" spike
    plt.text(age_idx, total + 1.5, f"+{shock_val:.1f}%", 
             ha='center', color='#e74c3c', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# SHOCK SCENARIO 2: 15% Increase in Mortgage Costs (Housing Shock)
df['Vulnerable_Housing'] = apply_economic_shock(df, "expenses", 0.15)
shock_summary = df.groupby('Age_Group')[['Is_At_Risk_Base', 'Vulnerable_Housing']].mean() * 100
plot_data = shock_summary.loc[[1, 2, 3, 4]]

# Chart 1: Bar chart of % newly vulnerable by age group after shock
# Visualize the % of each age group that became vulnerable after the shock
sns.barplot(
    x=['18-24', '25-34', '35-44', '45-54'], 
    y=plot_data['Vulnerable_Housing'], 
    palette='Blues_r'
)
plt.title('Age Group Breakdown: % Pushed into Risk by 15% Housing Shock', fontsize=13)
plt.ylabel('% of Group Newly Vulnerable')
plt.show()

# Chart 2: Stacked Bar Chart of Baseline Risk vs. New Risk After Shock
# Create a stacked bar chart comparing baseline risk vs. new risk after shock for each age group, to answer: "How much did the shock increase vulnerability in each age group?" 
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(plot_data.index, plot_data['Is_At_Risk_Base'], 
       label='Existing Risk (Baseline)', color="#69adbb", alpha=0.8)

ax.bar(plot_data.index, plot_data['Vulnerable_Housing'], 
       bottom=plot_data['Is_At_Risk_Base'], 
       label='New Risk (After 15% Shock)', color="#1650bb")

ax.set_xticks([1, 2, 3, 4]) # Set the position of the ticks
ax.set_xticklabels(['18-24', '25-34', '35-44', '45-54']) # Set the text labels

plt.title('Total Financial Vulnerability: Baseline vs. Post Housing Shock', fontsize=15, fontweight='bold')
plt.ylabel('% of Demographic At-Risk')
plt.xlabel('Age Group')
plt.ylim(0, 80)
plt.legend()

for i, age_idx in enumerate(plot_data.index):
    base_val = plot_data.loc[age_idx, 'Is_At_Risk_Base']
    shock_val = plot_data.loc[age_idx, 'Vulnerable_Housing']
    total = base_val + shock_val
    
    # Label for the "New Risk" spike
    plt.text(age_idx, total + 1.5, f"+{shock_val:.1f}%", 
             ha='center', color='#1650bb', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# SHOCK SCENARIO 3: 15% Decrease in Income (Recession/Job Loss Shock)
df['Vulnerable_Income'] = apply_economic_shock(df, "income", -0.15)
shock_summary = df.groupby('Age_Group')[['Is_At_Risk_Base', 'Vulnerable_Income']].mean() * 100
plot_data = shock_summary.loc[[1, 2, 3, 4]]

# Chart 1: Bar chart of % newly vulnerable by age group after shock
# Visualize the % of each age group that became vulnerable after the shock
sns.barplot(
    x=['18-24', '25-34', '35-44', '45-54'], 
    y=plot_data['Vulnerable_Income'], 
    palette='Greens_r'
)
plt.title('Age Group Breakdown: % Pushed into Risk by 15% Income Shock', fontsize=13)
plt.ylabel('% of Group Newly Vulnerable')
plt.show()

# Chart 2: Stacked Bar Chart of Baseline Risk vs. New Risk After Shock
# Create a stacked bar chart comparing baseline risk vs. new risk after shock for each age group, to answer: "How much did the shock increase vulnerability in each age group?" 
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(plot_data.index, plot_data['Is_At_Risk_Base'], 
       label='Existing Risk (Baseline)', color="#84E09B", alpha=0.8)

ax.bar(plot_data.index, plot_data['Vulnerable_Income'], 
       bottom=plot_data['Is_At_Risk_Base'], 
       label='New Risk (After 15% Shock)', color="#345e41")

ax.set_xticks([1, 2, 3, 4]) # Set the position of the ticks
ax.set_xticklabels(['18-24', '25-34', '35-44', '45-54']) # Set the text labels

plt.title('Total Financial Vulnerability: Baseline vs. Post Income Shock', fontsize=15, fontweight='bold')
plt.ylabel('% of Demographic At-Risk')
plt.xlabel('Age Group')
plt.ylim(0, 95)
plt.legend()

for i, age_idx in enumerate(plot_data.index):
    base_val = plot_data.loc[age_idx, 'Is_At_Risk_Base']
    shock_val = plot_data.loc[age_idx, 'Vulnerable_Income']
    total = base_val + shock_val
    
    # Label for the "New Risk" spike
    plt.text(age_idx, total + 1.5, f"+{shock_val:.1f}%", 
             ha='center', color='#1650bb', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# FEATURE IMPORTANCE ANALYSIS
from sklearn.inspection import permutation_importance
# 1. Define the features (The "Predictors")
features = ['Age_Group', 'Annual_After_Tax_Income', 'Total_Liquidity', 
            'Housing_Burden_Ratio', 'Student_Loan_Debt', 'Credit_Card_Debt', 'DTI_Ratio']

# 2. Prepare X and y
# We fill NaNs with 0 so the model doesn't crash
x = df[features].fillna(0)
y = df['Is_At_Risk_Base']

# 3. Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x, y)

# 4. Run Permutation Importance (The "Shuffle" Test)
# This evaluates how much accuracy drops when a feature is randomized
result = permutation_importance(model, x, y, n_repeats=10, random_state=42)

# 5. Create Importance DataFrame
perm_importance_df = pd.DataFrame({
    'feature': features,
    'importance': result.importances_mean
}).sort_values(by='importance', ascending=False)

# 6. Plot the results
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=perm_importance_df, palette='magma')

plt.title("Predictive Power of Financial Features (Permutation Test)", fontsize=16, fontweight='bold')
plt.xlabel("Impact on Model Accuracy (Higher = More Important)", fontsize=12)
plt.ylabel("Financial Feature")
plt.tight_layout()
plt.show()

In [ ]:
if "Liquidity_Months" not in df.columns:
    if "Years_of_Runway" in df.columns:
        df["Liquidity_Months"] = df["Years_of_Runway"] * 12
    else:
        raise KeyError("No Liquidity_Months or Years_of_Runway found. Run the resilience/feature engineering cell first.")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

persona_feats = ["Liquidity_Months","Housing_Burden_Ratio","DTI_Ratio","Annual_After_Tax_Income","Total_Liquidity"]

P = df[persona_feats].replace([np.inf,-np.inf], np.nan)
P = P.fillna(P.median(numeric_only=True))

P_scaled = StandardScaler().fit_transform(P)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
df["Persona"] = kmeans.fit_predict(P_scaled)

persona_summary = df.groupby("Persona")[persona_feats].median()
persona_summary["risk_%"] = df.groupby("Persona")["Is_At_Risk_Base"].mean() * 100
persona_summary.sort_values("risk_%", ascending=False)

In [ ]:
persona_colors = {
    0: "#B22222",   # dark red
    1: "#FF8C00",   # orange
    2: "#FFD700",   # gold
    3: "#4682B4",   # steel blue
    4: "#2E8B57"    # sea green
}

# First assign names based on sorted order
persona_summary = persona_summary.sort_values("risk_%", ascending=False).reset_index()

persona_names = [
    "Debt-Strained Renters",
    "Housing-Stressed Households",
    "Financially Fragile Middle",
    "Stable Earners",
    "Asset-Rich Resilient"
]

persona_summary["Persona_Name"] = persona_names[:len(persona_summary)]

# Define colors from high risk to low risk
colors = ["#B22222", "#FF8C00", "#FFD700", "#4682B4", "#2E8B57"]

plt.figure(figsize=(10,6))
plt.bar(persona_summary["Persona_Name"], persona_summary["risk_%"], color=colors[:len(persona_summary)])
plt.xticks(rotation=30, ha="right")
plt.ylabel("At-Risk Rate (%)")
plt.title("Financial Risk by Household Persona")
plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity Analysis: Stress Test Curves by Shock Severity: How is the entire population's vulnerability rate affected as we increase the severity of the shock?

def stress_test_curve(df, shock_grid=np.arange(0, 0.31, 0.05)):
    df = df.copy()

    # --- ensure required columns exist ---
    required = ["Total_Liquidity", "Annual_Expenses_Base", "Annual_After_Tax_Income",
                "Housing_Burden_Ratio", "DTI_Ratio", "Skipped_Payments"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns needed for stress test: {missing}")

    rates = []
    newly_vuln = []

    for s in shock_grid:
        temp = df.copy()

        # shocked expenses
        temp["Annual_Expenses_Shocked"] = temp["Annual_Expenses_Base"] * (1 + s)

        # runway in YEARS under shock
        temp["Runway_Post_Shock"] = temp["Total_Liquidity"] / temp["Annual_Expenses_Shocked"].replace(0, np.nan)

        # define shocked risk using SAME baseline logic style
        temp["Is_At_Risk_Shocked"] = np.where(
            (temp["Runway_Post_Shock"] < 0.25) |              # <3 months
            (temp["Housing_Burden_Ratio"] > 0.3) |
            (temp["DTI_Ratio"] > 0.50) |
            (temp["Skipped_Payments"] == 1),
            1, 0
        )

        # if baseline doesn't exist, compute a baseline version too
        if "Is_At_Risk_Base" not in temp.columns:
            temp["Is_At_Risk_Base"] = np.where(
                ((temp["Total_Liquidity"] / temp["Annual_Expenses_Base"].replace(0, np.nan)) < 0.25) |
                (temp["Housing_Burden_Ratio"] > 0.3) |
                (temp["DTI_Ratio"] > 0.50) |
                (temp["Skipped_Payments"] == 1),
                1, 0
            )

        temp["Became_Vulnerable"] = ((temp["Is_At_Risk_Base"] == 0) & (temp["Is_At_Risk_Shocked"] == 1)).astype(int)

        rates.append(temp["Is_At_Risk_Shocked"].mean() * 100)
        newly_vuln.append(temp["Became_Vulnerable"].mean() * 100)

    return shock_grid, rates, newly_vuln


shock_grid = np.arange(0, 0.31, 0.05)
shock_grid, rates, newly_vuln = stress_test_curve(df, shock_grid)

# The Total Risk Curve: % At-Risk vs. Shock Severity
plt.figure(figsize=(8,5))
plt.plot(shock_grid*100, rates, marker="o")
plt.xlabel("Expense Shock (%)")
plt.ylabel("At-Risk Rate (%)")
plt.title("Stress Test: At-Risk Rate vs Expense Shock")
plt.show()

# The Fragility Curve (Newly Vulnerable): % Newly Vulnerable vs. Shock Severity
plt.figure(figsize=(8,5))
plt.plot(shock_grid*100, newly_vuln, marker="o")
plt.xlabel("Expense Shock (%)")
plt.ylabel("Newly Vulnerable (%)")
plt.title("Fragility Curve: % Newly Vulnerable vs Shock")
plt.show()


In [ ]:
# Stress Test Curves by Age Group
age_labels = {1:"18-24", 2:"25-34", 3:"35-44", 4:"45-54"}

plt.figure(figsize=(9,6))
for ag in sorted(age_labels.keys()):
    if ag in df["Age_Group"].unique():
        sub = df[df["Age_Group"] == ag]
        _, r, _ = stress_test_curve(sub, shock_grid)
        plt.plot(shock_grid*100, r, marker="o", label=age_labels[ag], linewidth=2)
 
plt.xlabel("Expense Shock (%)")
plt.ylabel("At-Risk Rate (%)")
plt.title("Stress Test by Age Group")
plt.legend()
plt.show()